# 🏋️ Step 6 — Model Training & Evaluation Metrics
**File:** `05_model_training.ipynb`

**Input:**  `data/X_train.pkl`, `data/X_test.pkl`, `data/y_train.pkl`, `data/y_test.pkl`  ← notebook 04

**Output:** `models/emotion_model.pkl`  → for notebooks 06, 07, 08

In [ ]:
!pip install scikit-learn matplotlib seaborn joblib -q

In [ ]:
import joblib
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model    import LogisticRegression
from sklearn.ensemble        import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm             import SVC
from sklearn.metrics         import accuracy_score, f1_score, classification_report, confusion_matrix

# ── Load feature matrices ──
X_train = joblib.load('data/X_train.pkl')
X_test  = joblib.load('data/X_test.pkl')
y_train = joblib.load('data/y_train.pkl')
y_test  = joblib.load('data/y_test.pkl')
le      = joblib.load('models/label_encoder.pkl')

print(f'✅ Data loaded — Train: {X_train.shape} | Test: {X_test.shape}')

In [ ]:
# ── Train multiple models ──
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, C=1.0, solver='lbfgs'),
    'Random Forest':       RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'SVM (Linear)':        SVC(kernel='linear', C=1.0, probability=True),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=150, random_state=42)
}

results = {}
print('🚀 Training models...\n')

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc   = accuracy_score(y_test, preds)
    f1    = f1_score(y_test, preds, average='weighted')
    results[name] = {'accuracy': acc, 'f1': f1, 'model': model, 'preds': preds}
    print(f'  [{name}]  Accuracy: {acc:.4f} | F1: {f1:.4f}')

print('\n✅ All models trained!')

In [ ]:
# ── Select best model ──
best_name   = max(results, key=lambda k: results[k]['f1'])
best_result = results[best_name]
best_model  = best_result['model']
best_preds  = best_result['preds']

print(f'🏆 Best Model : {best_name}')
print(f'   Accuracy  : {best_result["accuracy"]:.4f}')
print(f'   F1 Score  : {best_result["f1"]:.4f}')
print('\n📋 Classification Report:')
print(classification_report(y_test, best_preds, target_names=le.classes_))

In [ ]:
# ── Visualize results ──
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

cm = confusion_matrix(y_test, best_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_, ax=axes[0])
axes[0].set_title(f'Confusion Matrix — {best_name}', fontweight='bold')
axes[0].set_ylabel('Actual'); axes[0].set_xlabel('Predicted')

names = list(results.keys())
accs  = [results[m]['accuracy'] for m in names]
f1s   = [results[m]['f1'] for m in names]
x     = np.arange(len(names))
axes[1].bar(x-0.175, accs, 0.35, label='Accuracy', color='#4D96FF', alpha=0.85)
axes[1].bar(x+0.175, f1s,  0.35, label='F1 Score',  color='#FF6B6B', alpha=0.85)
axes[1].set_xticks(x); axes[1].set_xticklabels(names, rotation=15, ha='right')
axes[1].set_ylim(0, 1.1); axes[1].set_title('Model Comparison', fontweight='bold')
axes[1].legend()

plt.tight_layout(); plt.show()

In [ ]:
# ── Save best model ──
joblib.dump(best_model, 'models/emotion_model.pkl')
print(f'✅ Saved best model → models/emotion_model.pkl')
print(f'   Model type: {type(best_model).__name__}')